# Clase 11 — Práctica: ETL y pipelines de datos

En la clase de teoría construimos un mini-pipeline ETL completo: Extract (datos crudos simulados), Transform (limpieza con pandas) y Load (`to_sql` a una tabla sqlite). Ahora vas a añadir una etapa de transformación adicional antes de la carga.

## Ejercicio: una etapa de transformación adicional

Partiendo del `datos_limpios` ya construido (Extract + la limpieza básica ya están hechos por ti), añade una columna derivada `importe_con_descuento` que aplique un 10% de descuento (`importe * 0.9`) **solo** a las filas donde `producto` sea `"Camiseta"`. El resto de importes deben quedar igual. Después, carga la tabla resultante en sqlite y compruébala con una consulta `SELECT`.

**Pistas:**
- Puedes usar indexado booleano de pandas: `datos_limpios.loc[condicion, "columna"]`.
- Empieza copiando la columna `importe` tal cual en la nueva columna, y luego sobrescribe solo las filas que cumplen la condición.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Extract (ya resuelto)
datos_crudos = pd.DataFrame({
    "id_venta":  [1, 2, 3, 4, 4, 5],
    "cliente":   ["Ana Torres", "luis peña", "CARLA RUIZ", "Diego Soto", "Diego Soto", None],
    "producto":  ["Camiseta", "camiseta", "Auriculares", None, None, "Mochila"],
    "importe":   [15.99, 15.99, 39.90, 25.00, 25.00, np.nan],
})

# Transform básico (ya resuelto)
datos_limpios = datos_crudos.drop_duplicates()
datos_limpios = datos_limpios.dropna(subset=["cliente"])
datos_limpios["importe"] = datos_limpios["importe"].fillna(0)
datos_limpios["cliente"] = datos_limpios["cliente"].str.title()
datos_limpios["producto"] = datos_limpios["producto"].fillna("Sin especificar").str.title()
datos_limpios["importe"] = pd.to_numeric(datos_limpios["importe"], errors="coerce").fillna(0)
datos_limpios = datos_limpios[datos_limpios["importe"] >= 0]

print("Datos limpios, antes de tu transformación adicional:")
print(datos_limpios)

In [ ]:
# TODO: crea la columna "importe_con_descuento" aplicando un 10%
# de descuento solo a las filas donde producto == "Camiseta".
# El resto de filas deben conservar el mismo importe original.

# datos_limpios["importe_con_descuento"] = ...

# print(datos_limpios)

In [ ]:
# TODO: carga datos_limpios (ya con la nueva columna) en una tabla
# "ventas" de una base de datos sqlite en memoria, y comprueba el
# resultado con una consulta SELECT.

# conexion = sqlite3.connect(':memory:')
# datos_limpios.to_sql("ventas", conexion, index=False, if_exists="replace")
# print(pd.read_sql_query("SELECT * FROM ventas", conexion))
# conexion.close()

<details>
<summary>💡 Ver solución</summary>

```python
# Nueva columna: por defecto, el mismo importe...
datos_limpios["importe_con_descuento"] = datos_limpios["importe"]

# ...y sobrescribimos solo las filas de "Camiseta" con el 10% de descuento.
es_camiseta = datos_limpios["producto"] == "Camiseta"
datos_limpios.loc[es_camiseta, "importe_con_descuento"] = datos_limpios.loc[es_camiseta, "importe"] * 0.9

print(datos_limpios)

# Load con la nueva columna ya incluida
conexion = sqlite3.connect(':memory:')
datos_limpios.to_sql("ventas", conexion, index=False, if_exists="replace")
print(pd.read_sql_query("SELECT * FROM ventas", conexion))
conexion.close()
```
</details>

## Para seguir

Vuelve al notebook de teoría si quieres repasar los tres pasos de ETL antes de intentar el ejercicio: [clase11_warehouses_etl_pipelines.ipynb](../notebooks_clase/clase11_warehouses_etl_pipelines.ipynb)